# Quasi-Experiment: Does Late Delivery Reduce Customer Satisfaction and Repeat Purchases?

**Research question:** Orders delivered after the estimated delivery date
(`is_late_delivery`) get lower review scores — but is that a causal
effect, or does lateness just correlate with other things (bigger orders,
farther states) that independently affect satisfaction?

This is a **quasi-experiment**: `is_late_delivery` was never randomly
assigned, so we have to check covariate balance and control for
confounders explicitly — unlike a true RCT, we can narrow the causal case
but not fully close it.

In [1]:
import duckdb
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest

con = duckdb.connect("../warehouse/olist.duckdb", read_only=True)

df = con.execute("""
    select o.review_score, o.is_late_delivery, o.item_count, o.order_total,
           o.freight_total, o.delivery_days, c.customer_state
    from main.fct_orders o
    join main.dim_customers c on o.customer_unique_id = c.customer_unique_id
    where o.order_status = 'delivered' and o.review_score is not null
""").fetchdf()

df.shape, df["is_late_delivery"].value_counts()

((95372, 7),
 is_late_delivery
 False    87759
 True      7613
 Name: count, dtype: int64)

## Covariate balance check

If `is_late_delivery` were randomly assigned, these groups would look
similar on covariates unrelated to the treatment. They don't.

In [2]:
balance = df.groupby("is_late_delivery").agg(
    n=("review_score", "size"),
    avg_item_count=("item_count", "mean"),
    avg_order_total=("order_total", "mean"),
    avg_freight_total=("freight_total", "mean"),
    avg_delivery_days=("delivery_days", "mean"),
).round(2)
balance

,n,avg_item_count,avg_order_total,avg_freight_total,avg_delivery_days
is_late_delivery,,,,,
False,87759,1.14,158.63,22.61,10.81
True,7613,1.11,171.47,24.52,31.34


## Naive comparison: review score, late vs. on-time

In [3]:
late = df[df.is_late_delivery]["review_score"]
ontime = df[~df.is_late_delivery]["review_score"]

t_stat, p_value = stats.ttest_ind(late, ontime, equal_var=False)

print(f"mean review score — late: {late.mean():.3f}, on-time: {ontime.mean():.3f}")
print(f"Welch's t-test: t = {t_stat:.2f}, p = {p_value:.2e}")

mean review score — late: 2.566, on-time: 4.296
Welch's t-test: t = -89.20, p = 0.00e+00


## Covariate-adjusted regression

Controlling for order size, freight, and state — does the gap shrink?

In [4]:
model = smf.ols(
    "review_score ~ is_late_delivery + item_count + order_total + freight_total + C(customer_state)",
    data=df,
).fit()

coef = model.params["is_late_delivery[T.True]"]
pval = model.pvalues["is_late_delivery[T.True]"]

print(f"Adjusted effect of late delivery on review score: {coef:.4f} (p = {pval:.2e})")
print(f"R-squared: {model.rsquared:.4f}, N = {int(model.nobs)}")
print()
print("Naive gap was -1.730 (4.296 - 2.566). Adjusted estimate barely moved —")
print("the covariates we have don't explain the effect, but unobserved")
print("confounders (product/seller quality, customer expectations) still could.")

Adjusted effect of late delivery on review score: -1.7155 (p = 0.00e+00)
R-squared: 0.1531, N = 95372

Naive gap was -1.730 (4.296 - 2.566). Adjusted estimate barely moved —
the covariates we have don't explain the effect, but unobserved
confounders (product/seller quality, customer expectations) still could.


## Repeat purchase: does a late delivery make customers leave?

The dataset ends 2018-10-17, so orders placed near the end haven't had
time to generate a "repeat" order yet — a **right-censoring** problem.
We restrict to orders placed before **2018-04-17** (~6 months of
follow-up for every included order) to avoid biasing the comparison.

In [5]:
repeat_df = con.execute("""
    with base as (
        select o.order_id, o.customer_unique_id, o.order_purchase_at, o.is_late_delivery
        from main.fct_orders o
        where o.order_status = 'delivered' and o.order_purchase_at < '2018-04-17'
    )
    select
        b.is_late_delivery,
        count(*) as n,
        sum(case when exists (
            select 1 from main.fct_orders o2
            where o2.customer_unique_id = b.customer_unique_id
              and o2.order_purchase_at > b.order_purchase_at
        ) then 1 else 0 end) as repeats
    from base b
    group by 1
""").fetchdf()

repeat_df["repeat_rate"] = (repeat_df["repeats"] / repeat_df["n"]).round(4)
repeat_df

,is_late_delivery,n,repeats,repeat_rate
0,False,61766,2373.0,0.0384
1,True,6110,177.0,0.0290


In [6]:
counts = repeat_df.sort_values("is_late_delivery", ascending=False)["repeats"].to_numpy()
nobs = repeat_df.sort_values("is_late_delivery", ascending=False)["n"].to_numpy()

z_stat, p_value = proportions_ztest(counts, nobs)
print(f"two-proportion z-test: z = {z_stat:.3f}, p = {p_value:.5f}")

two-proportion z-test: z = -3.706, p = 0.00021


## Why this isn't a true experiment

- **No randomization**: delivery lateness correlates with order size,
  freight cost, and customer state (RJ: 13.5% late vs. SP: 5.9% late) —
  proven by the covariate balance table above.
- **Adjusting for measured confounders barely changes the estimate**,
  which is *suggestive* the relationship is robust, but unobserved
  confounders (seller reliability, product quality, individual customer
  expectations) could still explain part of the gap — regression can only
  control for what we measured.
- **Right-censoring**: repeat-purchase analysis had to exclude the last
  ~6 months of orders, or recent on-time orders would look artificially
  "less loyal" just for lacking follow-up time.
- **Conclusion**: strong, statistically robust association between late
  delivery and both lower review scores and lower repeat-purchase rate.
  Treat as a well-supported *hypothesis* for a real experiment (e.g. a
  logistics-improvement test with randomized carrier assignment), not
  as proof of causation.